In [1]:
import pandas as pd
import numpy as np

# ── DATASET 1: campaign performance ──────────────────────────────────────────
performance = pd.DataFrame({
    'campaign_id':  ['C001','C002','C003','C004','C001','C002','C003','C004',
                     'C001','C002','C003','C004','C001','C002','C003','C004'],
    'date':         ['2024-01-01']*4 + ['2024-01-02']*4 + ['2024-01-03']*4 + ['2024-01-04']*4,
    'impressions':  [120000,95000,43000,76000, 134000,88000,51000,82000,
                     119000,91000,49000,78000, 141000,97000,0,80000],
    'clicks':       [1800,950,430,380, 2100,820,510,410,
                     1750,890,490,390, 2200,940,0,400],
    'conversions':  [90,20,86,19, 105,18,102,21, 88,22,98,20, 110,19,0,22],
    'spend':        [5400,2850,1290,1140, 6300,2460,1530,1230,
                     5250,2670,1470,1170, 6600,2820,0,1200]
})
performance['date'] = pd.to_datetime(performance['date'])

# ── DATASET 2: campaign metadata (separate lookup table) ─────────────────────
# C004 is in performance but NOT here (metadata was never entered)
# C005 is here but NOT in performance (new campaign, never ran)
campaigns = pd.DataFrame({
    'campaign_id':   ['C001','C002','C003','C005'],
    'campaign_name': ['Spring Sale','Brand Awareness','App Install','Summer Launch'],
    'advertiser':    ['Nike','Nike','Adidas','Adidas'],
    'channel':       ['display','social','social','display'],
    'budget':        [50000, 30000, 20000, 15000]
})

print('--- performance ---')
print(performance.head(8))
print('\n--- campaigns ---')
print(campaigns)


--- performance ---
  campaign_id       date  impressions  clicks  conversions  spend
0        C001 2024-01-01       120000    1800           90   5400
1        C002 2024-01-01        95000     950           20   2850
2        C003 2024-01-01        43000     430           86   1290
3        C004 2024-01-01        76000     380           19   1140
4        C001 2024-01-02       134000    2100          105   6300
5        C002 2024-01-02        88000     820           18   2460
6        C003 2024-01-02        51000     510          102   1530
7        C004 2024-01-02        82000     410           21   1230

--- campaigns ---
  campaign_id    campaign_name advertiser  channel  budget
0        C001      Spring Sale       Nike  display   50000
1        C002  Brand Awareness       Nike   social   30000
2        C003      App Install     Adidas   social   20000
3        C005    Summer Launch     Adidas  display   15000


In [12]:
#Problem 1
merged_df = performance.merge(campaigns,on='campaign_id', how= 'left', suffixes=('_perf','_camp'))

print(merged_df)

print(merged_df.shape)

print(merged_df[merged_df['campaign_name'].isna()])


   campaign_id       date  impressions  clicks  conversions  spend  \
0         C001 2024-01-01       120000    1800           90   5400   
1         C002 2024-01-01        95000     950           20   2850   
2         C003 2024-01-01        43000     430           86   1290   
3         C004 2024-01-01        76000     380           19   1140   
4         C001 2024-01-02       134000    2100          105   6300   
5         C002 2024-01-02        88000     820           18   2460   
6         C003 2024-01-02        51000     510          102   1530   
7         C004 2024-01-02        82000     410           21   1230   
8         C001 2024-01-03       119000    1750           88   5250   
9         C002 2024-01-03        91000     890           22   2670   
10        C003 2024-01-03        49000     490           98   1470   
11        C004 2024-01-03        78000     390           20   1170   
12        C001 2024-01-04       141000    2200          110   6600   
13        C002 2024-

In [35]:
#Problem 2
merged_df = merged_df.dropna(subset=['advertiser'])

aggregated = merged_df.groupby("advertiser").agg(total_spend=("spend","sum"),\
                                                  total_conversions = ("conversions", "sum")).reset_index()

aggregated['CPI'] = (aggregated["total_spend"]/aggregated["total_conversions"]).round(2)

print(aggregated)

  advertiser  total_spend  total_conversions    CPI
0     Adidas         4290                286  15.00
1       Nike        34350                472  72.78
